In [ ]:
# tool_04 Final 汇总到 thermol/Cu_100_absorption

## 目标
- 从 `1_body/Final`、`2_body/Final` 收集最终结构与结果文件
- 统一写入 `thermol/Cu_100_absorption/{1_Body,2_Body}`
- 生成 `address_manifest.tsv` 供后续直接索引，避免重复计算

In [ ]:
from pathlib import Path
import sys
import hashlib
import shutil
import csv

In [ ]:
# 加载 build/shared
_p = Path.cwd()
while _p != _p.parent:
    if (_p / 'shared' / 'config.py').exists():
        sys.path.insert(0, str(_p))
        break
    _p = _p.parent

from shared.load_config import load_config

cfg = load_config()
go_root = Path(cfg.GO_ROOT)
absorption_root = go_root.parent
print(f'GO_ROOT: {go_root}')
print(f'absorption_root: {absorption_root}')

In [ ]:
# ---------- 用户参数（先改这里）----------
source_absorption_root = absorption_root

# 目标目录：thermol 统一索引池
target_root = Path('/home/ucaqzh0/thermol/Cu_100_absorption')

# body 源目录 -> 目标目录名映射
body_map = {
    '1_body': '1_Body',
    '2_body': '2_Body',
}

# 提取模式：all / selected
extract_scope = 'all'
selected_relative_paths = []  # 例如 ['OH/case1/job_xxx.vasp']

# 默认提取 Final 中的结构与关键结果
include_patterns = ['*.vasp', 'OUTCAR', '*.OUTCAR']

# 覆盖策略：False=已有文件跳过；True=覆盖
overwrite_existing = False

# 是否清空旧清单并重写（True 建议）
rewrite_manifest = True

In [ ]:
def file_sha1(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha1()
    with path.open('rb') as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def should_include(path: Path, patterns: list[str]) -> bool:
    name = path.name
    for pat in patterns:
        if path.match(pat) or name == pat:
            return True
    return False


def collect_from_final(final_root: Path, target_body_root: Path):
    if not final_root.exists():
        print(f'⚠ 源目录不存在，跳过: {final_root}')
        return []

    files = [p for p in final_root.rglob('*') if p.is_file()]
    files = [p for p in files if should_include(p, include_patterns)]

    if extract_scope == 'selected':
        selected_set = set(selected_relative_paths)
        files = [p for p in files if str(p.relative_to(final_root)) in selected_set]

    records = []
    for src in sorted(files):
        rel = src.relative_to(final_root)
        dst = target_body_root / rel
        dst.parent.mkdir(parents=True, exist_ok=True)

        if dst.exists() and not overwrite_existing:
            copied = False
        else:
            shutil.copy2(src, dst)
            copied = True

        records.append({
            'source_path': str(src.resolve()),
            'target_path': str(dst.resolve()),
            'relative_final_path': str(rel),
            'file_name': src.name,
            'sha1': file_sha1(src),
            'copied': copied,
        })

    return records

In [ ]:
target_root.mkdir(parents=True, exist_ok=True)
manifest_path = target_root / 'address_manifest.tsv'

all_rows = []
for body_src, body_dst in body_map.items():
    final_root = Path(source_absorption_root) / body_src / 'Final'
    target_body_root = target_root / body_dst
    target_body_root.mkdir(parents=True, exist_ok=True)

    recs = collect_from_final(final_root, target_body_root)
    for r in recs:
        all_rows.append({
            'body': body_dst,
            **r,
        })

if rewrite_manifest or (not manifest_path.exists()):
    mode = 'w'
else:
    mode = 'a'

with manifest_path.open(mode, newline='', encoding='utf-8') as f:
    writer = csv.writer(f, delimiter='\t')
    if mode == 'w':
        writer.writerow([
            'record_id', 'body', 'source_path', 'target_path',
            'relative_final_path', 'file_name', 'sha1'
        ])
    for i, row in enumerate(all_rows, start=1):
        writer.writerow([
            i, row['body'], row['source_path'], row['target_path'],
            row['relative_final_path'], row['file_name'], row['sha1']
        ])

copied_count = sum(1 for r in all_rows if r['copied'])
skipped_count = len(all_rows) - copied_count
print('✅ 汇总完成')
print(f'- target_root: {target_root}')
print(f'- manifest: {manifest_path}')
print(f'- matched files: {len(all_rows)}')
print(f'- copied: {copied_count}, skipped(existing): {skipped_count}')

## 输出说明
- 源：`<absorption>/1_body/Final` 与 `<absorption>/2_body/Final`
- 目标：`/home/ucaqzh0/thermol/Cu_100_absorption/{1_Body,2_Body}`
- 清单：`/home/ucaqzh0/thermol/Cu_100_absorption/address_manifest.tsv`
- 清单中记录每个文件的源地址、目标地址与 `sha1`，便于去重与复用索引

In [ ]:
# 可选：按哈希查看重复文件（用于快速去重）
from collections import defaultdict

sha_map = defaultdict(list)
for row in all_rows:
    sha_map[row['sha1']].append(row['target_path'])

dups = {k: v for k, v in sha_map.items() if len(v) > 1}
print(f'duplicate sha groups: {len(dups)}')
for k, v in list(dups.items())[:10]:
    print(k, '->', len(v))
    for p in v:
        print('  ', p)